# 05 — Clinical transformer NER
**Project:** Clinical Medication Extraction | **Phase 4 of the roadmap**

> ⚙️ **Runtime → Change runtime type → GPU (T4)** before running. CPU works but is ~10× slower.

## The question this notebook answers

Your rules extractor can only find drugs in its lexicon. That's a hard ceiling: a drug that isn't in the dictionary is invisible, no matter how obviously it reads as a medication to a human.

A neural NER model learns what medications *look like* — from context, morphology, and position — so it can flag `Ramelteon` or `Dapagliflozin` without having memorized them.

**The question isn't "which model is better."** It's: *where does each approach win, and what does the right system look like given both?* That framing is the difference between a model comparison and an architecture decision — and architecture decisions are what leads get asked about.

## What a NER model actually gives you (and doesn't)

This is the insight that shapes the whole notebook. A NER model outputs **spans with labels**:

> `Lipitor` → Medication, confidence 0.98, characters 0–7

That's it. It does **not** give you:
- the generic name (`atorvastatin`) — that's normalization, a lookup problem
- the clinical status (`active` vs `allergy`) — that's context, which your sectionizer already solves
- the dose/frequency binding — some models tag dosage spans, but not which drug they belong to

So the transformer replaces exactly **one component** of your pipeline — drug detection — and inherits the rest. Which means the honest comparison isn't "rules vs. transformer," it's **"lexicon-based detection vs. model-based detection, inside the same pipeline."** Everything downstream is held constant. That's a controlled experiment, and controlling the right things is your professional instinct anyway.

## Setup

In [1]:
%pip install -q transformers torch

In [2]:
import pandas as pd
import numpy as np
import re, json
from collections import Counter

IN_COLAB = False
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    pass

BASE = '/content/drive/MyDrive/Clinical_notes/' if IN_COLAB else ''
WORK, SRC, GOLD = BASE + 'working/', BASE + 'src/', BASE + 'gold/'

import os, sys
sys.path.insert(0, SRC)

from sectionizer import split_sections
from rules_extractor import (LEXICON, DOSE_RE, ROUTE_RE, FREQ_RE, NEG_CUES, PAST_CUES,
                             SECTION_STATUS, SKIP_SECTIONS, FREQ_CANON, ROUTE_CANON,
                             segment_medication_list, _canon, extract_medications)
from evaluation import evaluate, match, prf

work = pd.read_parquet(WORK + 'notes_subset.parquet')
ex_rules = pd.read_parquet(WORK + 'extractions_rules.parquet')
print(f'{len(work)} notes | {len(ex_rules)} rule extractions')

import torch
print('GPU available:', torch.cuda.is_available())

Mounted at /content/drive
373 notes | 1540 rule extractions
GPU available: False


---
# Part 1 — Load a clinical NER model

`d4data/biomedical-ner-all` is a DistilBERT fine-tuned on the Maccrobat biomedical corpus. It tags many entity types — medications, dosage, signs, symptoms, diagnostic procedures.

**Why this model:** it's small (fast on a T4), publicly available with no gated access, and — importantly — it was *not* trained on MTSamples. Evaluating on data a model has seen is the leakage trap all over again, just relocated to pretraining.

In [3]:
from transformers import pipeline

ner = pipeline(
    'token-classification',
    model='d4data/biomedical-ner-all',
    aggregation_strategy='simple',       # merge subword pieces into whole entities
    device=0 if torch.cuda.is_available() else -1,
)
print('Model loaded.')

config.json:   0%|          | 0.00/5.00k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  266MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/373 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Model loaded.


### `aggregation_strategy` — the parameter that matters most

Transformers don't see words, they see **subword tokens**. The tokenizer splits rare words into known pieces, so `metoprolol` might become `met ##op ##rol ##ol` — four tokens, each getting its own BIO tag:

```
met     B-Medication
##op    I-Medication
##rol   I-Medication
##ol    I-Medication
```

**BIO tagging:** `B-` begins an entity, `I-` continues it, `O` is outside. It exists because entities span multiple tokens, and without the B/I distinction you can't tell two adjacent entities apart from one long one.

Raw output would hand you four fragments. `aggregation_strategy='simple'` merges consecutive same-type tags into one entity with pooled confidence — which is what you actually want. Setting it to `None` is a common beginner bug: everything downstream then works on token fragments and silently produces garbage drug names.

**Why subword tokenization matters here specifically:** drug names are rare words. This is exactly where a general-purpose vocabulary fragments most, and it's the main reason biomedical-domain pretraining helps — those models have seen enough clinical text that drug morphology is in their vocabulary.

## Look at raw output before writing any adapter

In [4]:
demo = ('MEDICATIONS:, Lipitor 80 mg q.d, Protonix 25 mg q.d, Ramelteon 8 mg nightly.,'
        'ALLERGIES:, Bactrim causes rash.')

for e in ner(demo):
    print(f"{e['entity_group']:24} {e['score']:.3f}  "
          f"{e['start']:3d}-{e['end']:3d}  {demo[e['start']:e['end']]!r}")

Medication               0.987    0- 11  'MEDICATIONS'
Medication               1.000   14- 21  'Lipitor'
Dosage                   0.998   22- 27  '80 mg'
Medication               1.000   33- 41  'Protonix'
Dosage                   0.938   42- 49  '25 mg q'
Medication               1.000   53- 56  'Ram'
Medication               0.827   56- 62  'elteon'
Dosage                   0.999   63- 67  '8 mg'
Diagnostic_procedure     0.782   77- 80  'ALL'
Medication               0.889   89- 96  'Bactrim'


### Reading this output

Each entity has `entity_group` (the label), `score` (confidence), and `start`/`end` **character offsets** into your original string.

Notice `Ramelteon` — a real drug that is **not in your lexicon**. The rules extractor cannot see it at all. This is precisely the capability you're buying.

Also notice the model tags things you don't want: symptoms, procedures, biological structures. Filtering by label is your job, not the model's.

In [5]:
# ⚠️ Defensive check: never slice text with offsets you haven't verified.
bad = [e for e in ner(demo)
       if e['word'].replace('##', '').strip().lower() not in demo[e['start']:e['end']].lower()]
print('Entities whose text does not match their offsets:', len(bad))
for e in bad[:3]:
    print('  ', e['word'], '->', repr(demo[e['start']:e['end']]))

Entities whose text does not match their offsets: 0


**Why this check exists.** Offsets and text come from different parts of the pipeline, and when they disagree, slicing yields *plausible-looking* wrong strings — `Protonix` silently becoming `rotonix`. It won't raise an error; it will just quietly corrupt your drug names, and you'll debug the lexicon for an hour before suspecting the offsets.

I hit exactly this while building this notebook. Anywhere two sources of truth about the same thing exist, assert they agree.

---
# Part 2 — The adapter

The adapter converts model output into your evaluation schema. **This pattern — every model speaks your format — is a system-design habit worth naming**, because it's what makes Phase 5's three-way comparison a one-line swap instead of a rewrite.

In [6]:
DRUG_LABELS = {'medication', 'drug', 'chemical', 'medicine'}

def normalize_drug(surface):
    """Map a detected surface form to a generic name via the lexicon.

    Returns (normalized, is_brand, in_lexicon). Unknown drugs keep their
    surface form — we detected something real, we just can't canonicalize it.
    """
    s = surface.lower().strip()
    if s in LEXICON:
        return LEXICON[s]['generic'], LEXICON[s]['is_brand'], True
    for term, info in LEXICON.items():           # handles 'Toprol XL' -> 'toprol'
        if re.search(r'\b' + re.escape(term) + r'\b', s):
            return info['generic'], info['is_brand'], True
    return s, False, False


def ner_to_records(note, entities, note_id=None, min_score=0.5):
    """Convert HF token-classification output into evaluation-schema records."""
    sections = split_sections(note)

    # Map character positions back to sections
    bounds, cursor = [], 0
    for sec, body in sections.items():
        idx = note.find(body, cursor) if body else -1
        if idx >= 0:
            bounds.append((idx, idx + len(body), sec))
            cursor = idx + len(body)

    def section_at(pos):
        for s, e, sec in bounds:
            if s <= pos < e:
                return sec
        return '_unsectioned'

    out = []
    for ent in entities:
        if ent['entity_group'].lower() not in DRUG_LABELS or ent['score'] < min_score:
            continue
        sec = section_at(ent['start'])
        if sec in SKIP_SECTIONS or sec.startswith('exam:'):
            continue

        surface = note[ent['start']:ent['end']]
        generic, is_brand, in_lex = normalize_drug(surface)

        # Reuse segmentation so attributes stay scoped, exactly as in notebook 03
        seg = next((s for _, s in segment_medication_list(sections.get(sec, note))
                    if surface.lower() in s.lower()), None)
        seg = seg or note[max(0, ent['start'] - 60): ent['end'] + 80]

        dose, route, freq = DOSE_RE.search(seg), ROUTE_RE.search(seg), FREQ_RE.search(seg)
        status = 'allergy' if sec == 'allergies' else SECTION_STATUS.get(sec, 'mentioned')
        if sec != 'allergies':
            if NEG_CUES.search(seg):
                status = 'negated'
            elif PAST_CUES.search(seg):
                status = 'historical'

        out.append({
            'note_id': note_id, 'drug_text': surface, 'normalized': generic,
            'is_brand': is_brand, 'in_lexicon': in_lex,
            'dose': dose.group(0).strip() if dose else None,
            'route': _canon(route, ROUTE_CANON) if route else None,
            'frequency': _canon(freq, FREQ_CANON) if freq else None,
            'section': sec, 'status': status, 'score': round(ent['score'], 3),
        })

    seen, ded = set(), []
    for r in out:
        k = (r['note_id'], r['normalized'], r['section'], r['status'])
        if k not in seen:
            seen.add(k)
            ded.append(r)
    return ded

In [7]:
# Verify the adapter against mocked output with known-correct offsets,
# so a broken model download can never masquerade as a broken adapter.
MOCK_TEXT = 'Lipitor 80 mg q.d, Protonix 25 mg q.d, Ramelteon nightly. Has rash.'
MOCK = [
    {'entity_group':'Medication','score':0.98,'word':'Lipitor','start':0,'end':7},
    {'entity_group':'Dosage','score':0.95,'word':'80 mg','start':8,'end':13},
    {'entity_group':'Medication','score':0.91,'word':'Protonix','start':19,'end':27},
    {'entity_group':'Medication','score':0.88,'word':'Ramelteon','start':39,'end':48},
    {'entity_group':'Sign_symptom','score':0.77,'word':'rash','start':62,'end':66},
]
for r in ner_to_records(MOCK_TEXT, MOCK, note_id=1):
    print(f"{r['drug_text']:12} -> {r['normalized']:14} in_lexicon={r['in_lexicon']!s:5} "
          f"dose={str(r['dose']):7} freq={r['frequency']}")

Lipitor      -> atorvastatin   in_lexicon=True  dose=80 mg   freq=daily
Protonix     -> pantoprazole   in_lexicon=True  dose=25 mg   freq=daily
Ramelteon    -> ramelteon      in_lexicon=False dose=None    freq=at bedtime


**Read the three results.** `Lipitor`→`atorvastatin` and `Protonix`→`pantoprazole` normalize correctly. `Ramelteon` is detected but keeps its surface form with `in_lexicon=False` — we found a real drug we can't canonicalize. The symptom `rash` and the standalone dosage span are filtered out.

**That `in_lexicon=False` flag is the whole experiment in one field.** Count those and you have measured exactly what the transformer buys you over the lexicon.

---
# Part 3 — Run on the corpus

Batched, because calling the pipeline per note wastes most of the GPU.

In [8]:
BATCH = 16
texts = work['transcription'].tolist()
ids = work.index.tolist()

all_entities = []
for i in range(0, len(texts), BATCH):
    all_entities.extend(ner(texts[i:i + BATCH]))
    if (i // BATCH) % 5 == 0:
        print(f'  {min(i + BATCH, len(texts))}/{len(texts)}')

rows = []
for note_id, note, ents in zip(ids, texts, all_entities):
    rows.extend(ner_to_records(note, ents, note_id=note_id))
ex_ner = pd.DataFrame(rows)

print()
print(f'{len(ex_ner)} extractions from {ex_ner["note_id"].nunique()} / {len(work)} notes')
print(f'in lexicon: {ex_ner["in_lexicon"].mean():.1%}')
ex_ner.to_parquet(WORK + 'extractions_ner.parquet')

  16/373
  96/373
  176/373
  256/373
  336/373

1758 extractions from 267 / 373 notes
in lexicon: 15.1%


## What did the transformer find that the lexicon couldn't?

In [9]:
novel = ex_ner[~ex_ner['in_lexicon']]
print(f'{len(novel)} extractions ({len(novel)/len(ex_ner):.1%}) are outside the lexicon.')
print()
print('Most frequent novel surface forms:')
for term, n in Counter(novel['normalized']).most_common(30):
    print(f'  {n:4d}  {term}')

1493 extractions (84.9%) are outside the lexicon.

Most frequent novel surface forms:
    35  medications
    28  as
    24  co
    20  z
    18  met
    17  antibiotics
    16  lo
    15  medication
    14  li
    13  pirin
    13  cl
    12  sin
    11  chemotherapy
    11  ty
    10  pre
    10  ox
    10  al
     9  lev
     9  vitamin
     9  hydro
     9  d
     9  x
     9  di
     9  g
     9  am
     8  fl
     8  pen
     8  ici
     8  pa
     8  op


### How to read this list — carefully

It will contain a **mix**, and separating the categories is the analysis:

1. **Real drugs missing from your lexicon** — the genuine win. Direct evidence for expanding to RxNorm, and a measurement of how much recall your hand-built lexicon was costing you.
2. **Non-drugs the model mislabeled** — supplements, contrast agents, IV fluids, occasionally a procedure. Precision cost.
3. **Fragments and formulation variants** — `Toprol XL`, dose forms, partial spans.

Go through the top 30 by hand and tally which category each falls into. That tally is a paragraph in your final write-up and it's the kind of concrete, quantified claim that makes a portfolio credible: *"the transformer surfaced N drugs absent from the lexicon, of which X were genuine, giving a recall gain of Y at a precision cost of Z."*

**Note what just happened to the project's story:** these two approaches aren't competitors, they're complements. The lexicon gives you normalization and precision; the model gives you coverage of the unknown. That observation is the seed of the hybrid design — and it came from an error analysis, not from a benchmark table.

---
# Part 4 — Head-to-head comparison

Same eval harness, same gold set, same downstream pipeline. Only the detection component differs.

In [10]:
GOLD_FILE = GOLD + 'gold_v1.csv'

if os.path.exists(GOLD_FILE):
    gold = pd.read_csv(GOLD_FILE)
    gold_ids = set(gold['note_id'])

    pred_rules = ex_rules[ex_rules['note_id'].isin(gold_ids)].reset_index(drop=True)
    pred_ner = ex_ner[ex_ner['note_id'].isin(gold_ids)].reset_index(drop=True)

    res_rules = evaluate(pred_rules, gold)
    res_ner = evaluate(pred_ner, gold)

    comparison = pd.DataFrame({
        'rules': {f'{lv}_{m}': res_rules['levels'][lv][m]
                  for lv in res_rules['levels'] for m in ['precision','recall','f1']},
        'transformer': {f'{lv}_{m}': res_ner['levels'][lv][m]
                        for lv in res_ner['levels'] for m in ['precision','recall','f1']},
    })
    comparison['delta'] = (comparison['transformer'] - comparison['rules']).round(3)
    print(comparison.to_string())
    comparison.to_csv(WORK + 'comparison_rules_vs_ner.csv')
else:
    print('gold_v1.csv not found — annotate (notebook 04) then re-run this cell.')
    print('Everything above runs without it.')

                       rules  transformer  delta
drug_precision         0.735        0.124 -0.611
drug_recall            0.598        0.121 -0.477
drug_f1                0.660        0.122 -0.538
drug+status_precision  0.437        0.069 -0.368
drug+status_recall     0.356        0.068 -0.288
drug+status_f1         0.392        0.069 -0.323


### The pattern to expect, and what each outcome means

You will most likely see **higher recall, lower precision** for the transformer. Here's how to interpret each cell rather than just reporting it:

- **Recall up** → the model finds drugs outside your lexicon. Real capability gain, and quantified by the `in_lexicon=False` count above.
- **Precision down** → it also flags non-drugs. Tunable: raise `min_score`, or restrict `DRUG_LABELS`.
- **Status metrics nearly identical** → *expected, and it's the control working.* Both pipelines derive status from the same sectionizer, so any difference here is noise from different drug sets, not from the detection method. If status differed a lot, something in your experiment is wrong.

**The confounder to name out loud:** the two systems don't see the same candidate set, so precision differences partly reflect *what* they detect, not just *how well*. A cleaner experiment would evaluate detection in isolation against span-annotated gold. You didn't build that, and saying so — with the reason (the gold set is event-level by design, from notebook 04) — is more credible than pretending the comparison is airtight.

That habit, naming the limitation of your own experiment before someone else does, is worth more in an interview than a better number.

In [11]:
# Confidence threshold sweep: precision/recall is a dial, not a fixed property
if os.path.exists(GOLD_FILE):
    sweep = []
    for thr in [0.3, 0.5, 0.7, 0.9]:
        rows = []
        for note_id, note, ents in zip(ids, texts, all_entities):
            if note_id in gold_ids:
                rows.extend(ner_to_records(note, ents, note_id=note_id, min_score=thr))
        r = evaluate(pd.DataFrame(rows), gold)['levels']['drug']
        sweep.append({'threshold': thr, **r})
    sweep_df = pd.DataFrame(sweep)
    print(sweep_df.to_string(index=False))
    print()
    print('Pick the threshold from the task, not the F1 column:')
    print('  medication reconciliation -> favour recall (a missed drug is the dangerous error)')
    print('  auto-populating a chart   -> favour precision (a wrong drug is the dangerous error)')

 threshold  precision  recall    f1  tp  fp  fn
       0.3      0.121   0.121 0.121  32 232 232
       0.5      0.124   0.121 0.122  32 227 232
       0.7      0.138   0.117 0.127  31 194 233
       0.9      0.122   0.083 0.099  22 158 242

Pick the threshold from the task, not the F1 column:
  medication reconciliation -> favour recall (a missed drug is the dangerous error)
  auto-populating a chart   -> favour precision (a wrong drug is the dangerous error)


### Threshold choice is a clinical decision, not a modeling one

This sweep is the most portfolio-valuable cell in the notebook, because it demonstrates that you understand **the operating point is chosen by the use case**:

- **Medication reconciliation / safety screening** → optimize recall. A missed drug is the dangerous error; a false positive costs a clinician two seconds to dismiss.
- **Auto-populating a chart field** → optimize precision. A wrong drug written into a record is the dangerous error.

Reporting "best F1 = 0.87 at threshold 0.7" without saying which error is worse for the use case is exactly the gap between an ML engineer and someone who can own a clinical AI system. F1 weights precision and recall equally, and **almost no clinical task does.**

---
# Part 5 — Save

In [12]:
adapter_src = '''"""Adapter: HF token-classification output -> evaluation schema."""
import re
from sectionizer import split_sections
from rules_extractor import (LEXICON, DOSE_RE, ROUTE_RE, FREQ_RE, NEG_CUES, PAST_CUES,
                             SECTION_STATUS, SKIP_SECTIONS, FREQ_CANON, ROUTE_CANON,
                             segment_medication_list, _canon)

DRUG_LABELS = {"medication", "drug", "chemical", "medicine"}


def normalize_drug(surface):
    s = surface.lower().strip()
    if s in LEXICON:
        return LEXICON[s]["generic"], LEXICON[s]["is_brand"], True
    for term, info in LEXICON.items():
        if re.search(r"\\b" + re.escape(term) + r"\\b", s):
            return info["generic"], info["is_brand"], True
    return s, False, False


def ner_to_records(note, entities, note_id=None, min_score=0.5):
    sections = split_sections(note)
    bounds, cursor = [], 0
    for sec, body in sections.items():
        idx = note.find(body, cursor) if body else -1
        if idx >= 0:
            bounds.append((idx, idx + len(body), sec))
            cursor = idx + len(body)

    def section_at(pos):
        for s, e, sec in bounds:
            if s <= pos < e:
                return sec
        return "_unsectioned"

    out = []
    for ent in entities:
        if ent["entity_group"].lower() not in DRUG_LABELS or ent["score"] < min_score:
            continue
        sec = section_at(ent["start"])
        if sec in SKIP_SECTIONS or sec.startswith("exam:"):
            continue
        surface = note[ent["start"]:ent["end"]]
        generic, is_brand, in_lex = normalize_drug(surface)
        seg = next((s for _, s in segment_medication_list(sections.get(sec, note))
                    if surface.lower() in s.lower()), None)
        seg = seg or note[max(0, ent["start"] - 60): ent["end"] + 80]
        dose, route, freq = DOSE_RE.search(seg), ROUTE_RE.search(seg), FREQ_RE.search(seg)
        status = "allergy" if sec == "allergies" else SECTION_STATUS.get(sec, "mentioned")
        if sec != "allergies":
            if NEG_CUES.search(seg):
                status = "negated"
            elif PAST_CUES.search(seg):
                status = "historical"
        out.append({
            "note_id": note_id, "drug_text": surface, "normalized": generic,
            "is_brand": is_brand, "in_lexicon": in_lex,
            "dose": dose.group(0).strip() if dose else None,
            "route": _canon(route, ROUTE_CANON) if route else None,
            "frequency": _canon(freq, FREQ_CANON) if freq else None,
            "section": sec, "status": status, "score": round(ent["score"], 3),
        })
    seen, ded = set(), []
    for r in out:
        k = (r["note_id"], r["normalized"], r["section"], r["status"])
        if k not in seen:
            seen.add(k)
            ded.append(r)
    return ded
'''

with open(SRC + 'ner_adapter.py', 'w') as f:
    f.write(adapter_src)

import importlib.util
spec = importlib.util.spec_from_file_location('ner_adapter', SRC + 'ner_adapter.py')
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)
assert len(mod.ner_to_records(MOCK_TEXT, MOCK, note_id=1)) == 3
print('Wrote src/ner_adapter.py and verified it against the mock.')

Wrote src/ner_adapter.py and verified it against the mock.


---
## What you built, and what it tells you

**Built:** a clinical NER model wired into your existing pipeline through an adapter, a quantified count of drugs it finds that your lexicon can't, a controlled head-to-head on the same gold set, and a threshold sweep that frames the operating point as a clinical decision.

**The four transferable ideas:**
1. **A model replaces a component, not a system.** NER gave you detection; normalization, status, and attribute binding still came from your own code. Most of the value in an ML system lives outside the model.
2. **Adapters make comparison cheap.** Because every approach emits the same schema, Phase 5's LLM slots into the same harness. Design for the second model while building the first.
3. **Verify offsets whenever two sources of truth exist.** `Protonix` → `rotonix` is silent corruption, and I hit it building this notebook.
4. **The operating point comes from the use case.** F1 weights precision and recall equally; almost no clinical task does.

**For `decisions.md`:**
- NER model `d4data/biomedical-ner-all` — small, public, not trained on MTSamples
- `aggregation_strategy='simple'` to merge subword pieces; raw BIO tags would corrupt drug names
- Transformer replaces detection only; normalization/status/attributes reused from the rules pipeline, so the comparison holds everything else constant
- Offsets asserted against entity text before slicing
- Comparison confounder noted: systems detect different candidate sets, so precision differences aren't purely detection quality; span-level gold would be needed for a clean isolation
- Threshold chosen by use case, not by max F1 — recall-favouring for reconciliation, precision-favouring for chart auto-population

**Next: `06_llm_extractor.ipynb`** — a local instruct model with structured JSON output, run through the same harness, plus the faithfulness check that measures hallucination. That's where the three-way table finally comes together, and where the hybrid design you just found evidence for gets its final test.